# Abelian Basis Sectors

This notebook replaces the old `AbelianBasisSectors.jl` script. It focuses on one practical question: how different symmetry labels combine, and how the split sectors recombine into the parent sector spectrum.

In [ ]:
import Pkg

function find_repo_root(start = pwd())
    dir = abspath(start)
    while true
        if isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "EDKit.jl"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate EDKit.jl repo root from $(start)")
        dir = parent
    end
end

const DEV = true
const REPO_ROOT = find_repo_root()
Pkg.activate(REPO_ROOT)
if DEV
    pushfirst!(LOAD_PATH, REPO_ROOT)
end

using EDKit
using LinearAlgebra
using Statistics


## Compare a parent sector with its parity split

We use a translation-invariant XX chain at half filling and momentum `k = 0`. In that setting, reflection parity is compatible with momentum, so the `(N, k)` sector can be decomposed into `p = ±1` subsectors.

In [ ]:
L = 8
target_N = L ÷ 2
target_k = 0
bond = spin((1.0, "xx"), (1.0, "yy"))

full_sector = basis(L = L, N = target_N, k = target_k)
full_vals = sort(eigvals(Hermitian(trans_inv_operator(bond, 2, full_sector))))

split_vals = Float64[]
split_dims = Dict{Int, Int}()
for parity in (1, -1)
    sector = basis(L = L, N = target_N, k = target_k, p = parity)
    split_dims[parity] = size(sector, 1)
    if !iszero(size(sector, 1))
        append!(split_vals, eigvals(Hermitian(trans_inv_operator(bond, 2, sector))))
    end
end
split_vals = sort(split_vals)

summary = (
    dim_parent = size(full_sector, 1),
    dim_even = split_dims[1],
    dim_odd = split_dims[-1],
    recombination_error = norm(full_vals - split_vals),
)
summary


## Plot the sector dimensions

Dimension counting is one of the fastest sanity checks when you start combining symmetries. The recombination error above checks the spectrum, while this plot checks the bookkeeping.

In [ ]:
if isdefined(Main, :IJulia)
    using Plots
    bar(["(N,k)", "(N,k,p=+1)", "(N,k,p=-1)"], [size(full_sector, 1), split_dims[1], split_dims[-1]]; ylabel = "dimension", label = nothing, title = "Sector dimensions")
else
    @info "Skipping Plots outside IJulia; returning dimensions instead"
    [size(full_sector, 1), split_dims[1], split_dims[-1]]
end


## Takeaway

`basis(...)` is most useful when you treat it as a bookkeeping tool for physically compatible symmetries. This small example shows the two checks that matter most in practice: dimensions should add up correctly, and the merged child-sector spectrum should reproduce the parent-sector spectrum.